<a href="https://colab.research.google.com/github/anubha06/uncertainty-aware-portfolio-optimization/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install yfinance --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

os.makedirs("data", exist_ok=True)
os.makedirs("plots", exist_ok=True)

print("Setup complete.")

Setup complete.


In [4]:
tickers = [
    # Technology
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "AMD", "INTC", "ORCL", "CRM",
    "ADBE", "CSCO", "QCOM", "IBM", "TXN",

    # Financials
    "JPM", "BAC", "WFC", "GS", "MS", "C", "AXP", "BLK", "SCHW", "USB",

    # Healthcare
    "JNJ", "UNH", "PFE", "MRK", "ABBV", "TMO", "ABT", "LLY", "BMY", "CVS",

    # Consumer Defensive
    "PG", "KO", "PEP", "WMT", "COST", "MDLZ", "CL", "KMB", "GIS",

    # Consumer Cyclical
    "TSLA", "HD", "LOW", "NKE", "SBUX", "MCD", "TGT", "BKNG", "TJX", "GM",

    # Energy
    "XOM", "CVX", "COP", "SLB", "EOG", "MPC", "PSX", "VLO", "OXY", "HAL",

    # Industrials
    "BA", "CAT", "GE", "HON", "UPS", "RTX", "LMT", "DE", "MMM", "FDX",

    # Communication
    "NFLX", "DIS", "CMCSA", "VZ", "T", "TMUS", "EA", "FOXA", "CHTR",

    # Utilities / Real Estate / Materials
    "NEE", "DUK", "SO", "AEP", "D", "AMT", "PLD", "SPG", "LIN", "APD",

    # Benchmark
    "SPY"
]

start_date = "2015-01-01"
end_date = "2024-12-31"

data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=True
)

prices = data["Close"]

# Remove stocks with too many missing values
missing_ratio = prices.isna().mean()
valid_tickers = missing_ratio[missing_ratio < 0.10].index.tolist()

prices = prices[valid_tickers]

# Fill small missing gaps
prices = prices.ffill().bfill()

# Calculate returns
simple_returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

# Save files again
prices.to_csv("data/prices_large_clean.csv")
simple_returns.to_csv("data/simple_returns_large.csv")
log_returns.to_csv("data/log_returns_large.csv")

print("Data recreated successfully.")
print("Prices shape:", prices.shape)
print("Returns shape:", simple_returns.shape)
print("Missing values:", prices.isna().sum().sum())

[*********************100%***********************]  94 of 94 completed


Data recreated successfully.
Prices shape: (2515, 93)
Returns shape: (2514, 93)
Missing values: 0


In [5]:
prices = pd.read_csv("data/prices_large_clean.csv", index_col=0, parse_dates=True)
simple_returns = pd.read_csv("data/simple_returns_large.csv", index_col=0, parse_dates=True)

print("Prices shape:", prices.shape)
print("Returns shape:", simple_returns.shape)

prices.head()

Prices shape: (2515, 93)
Returns shape: (2514, 93)


,AAPL,ABBV,ABT,ADBE,AEP,AMD,AMT,AMZN,APD,AXP,...,TSLA,TXN,UNH,UPS,USB,VLO,VZ,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,24.192606,41.411907,36.009220,72.339996,40.750809,2.67,75.751198,15.4260,102.235542,79.040199,...,14.620667,39.102745,83.828789,72.166580,30.177570,32.473061,25.732983,39.736126,23.127544,57.145561
2015-01-05,23.511061,40.632580,36.017227,71.980003,40.151051,2.66,74.656769,15.1095,98.966675,76.949921,...,14.006000,38.495892,82.447998,70.721695,29.450575,31.137764,25.519272,38.646454,23.060232,55.581963
2015-01-06,23.513273,40.431450,35.608223,70.529999,40.370953,2.63,74.459167,14.7645,98.931534,75.309982,...,14.085333,37.859764,82.281609,70.257507,29.060141,30.757156,25.776815,37.840122,23.237934,55.286453
2015-01-07,23.842978,42.065552,35.896919,71.110001,40.944065,2.58,75.242004,14.9210,100.000084,76.954758,...,14.063333,38.539745,83.121735,70.911285,29.315943,30.518484,25.610477,38.065315,23.854488,55.846653
2015-01-08,24.759081,42.505501,36.634758,72.919998,41.297279,2.61,75.941208,15.0230,102.312912,78.045601,...,14.041333,39.168549,87.089432,72.186226,29.531349,32.266640,26.159401,38.907986,24.357964,56.776196


In [6]:
def create_features_for_stock(ticker, prices, returns):
    """
    Creates ML features for one stock.
    Each row represents one stock on one date.
    Target is next-day return.
    """

    df = pd.DataFrame(index=returns.index)

    # Identity column
    df["ticker"] = ticker

    # Current daily return
    df["return_1d"] = returns[ticker]

    # Lag returns: previous returns
    df["lag_return_1d"] = returns[ticker].shift(1)
    df["lag_return_3d"] = returns[ticker].shift(3)
    df["lag_return_5d"] = returns[ticker].shift(5)
    df["lag_return_10d"] = returns[ticker].shift(10)

    # Rolling average returns
    df["rolling_mean_5d"] = returns[ticker].rolling(window=5).mean().shift(1)
    df["rolling_mean_10d"] = returns[ticker].rolling(window=10).mean().shift(1)
    df["rolling_mean_20d"] = returns[ticker].rolling(window=20).mean().shift(1)

    # Rolling volatility
    df["rolling_vol_5d"] = returns[ticker].rolling(window=5).std().shift(1)
    df["rolling_vol_10d"] = returns[ticker].rolling(window=10).std().shift(1)
    df["rolling_vol_20d"] = returns[ticker].rolling(window=20).std().shift(1)

    # Momentum features
    df["momentum_5d"] = prices[ticker].pct_change(periods=5).shift(1)
    df["momentum_10d"] = prices[ticker].pct_change(periods=10).shift(1)
    df["momentum_20d"] = prices[ticker].pct_change(periods=20).shift(1)

    # Moving average ratio
    ma_5 = prices[ticker].rolling(window=5).mean().shift(1)
    ma_20 = prices[ticker].rolling(window=20).mean().shift(1)
    df["ma_ratio_5_20"] = (ma_5 / ma_20) - 1

    # Target: tomorrow's return
    df["target_next_day_return"] = returns[ticker].shift(-1)

    # Move date from index to column
    df = df.reset_index()
    df = df.rename(columns={"Date": "date", "index": "date"})

    return df

In [7]:
all_features = []

for ticker in simple_returns.columns:
    stock_features = create_features_for_stock(ticker, prices, simple_returns)
    all_features.append(stock_features)

features_df = pd.concat(all_features, ignore_index=True)

print("Raw feature dataset shape:", features_df.shape)
features_df.head()

Raw feature dataset shape: (233802, 18)


,date,ticker,return_1d,lag_return_1d,lag_return_3d,lag_return_5d,lag_return_10d,rolling_mean_5d,rolling_mean_10d,rolling_mean_20d,rolling_vol_5d,rolling_vol_10d,rolling_vol_20d,momentum_5d,momentum_10d,momentum_20d,ma_ratio_5_20,target_next_day_return
0,2015-01-05,AAPL,-0.028172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000094
1,2015-01-06,AAPL,0.000094,-0.028172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014022
2,2015-01-07,AAPL,0.014022,0.000094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.038422
3,2015-01-08,AAPL,0.038422,0.014022,-0.028172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.001073
4,2015-01-09,AAPL,0.001073,0.038422,0.000094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.024641


In [8]:
features_df = features_df.dropna().reset_index(drop=True)

print("Clean feature dataset shape:", features_df.shape)
print("Number of unique tickers:", features_df["ticker"].nunique())
print("Start date:", features_df["date"].min())
print("End date:", features_df["date"].max())

features_df.head()

Clean feature dataset shape: (231849, 18)
Number of unique tickers: 93
Start date: 2015-02-03 00:00:00
End date: 2024-12-27 00:00:00


,date,ticker,return_1d,lag_return_1d,lag_return_3d,lag_return_5d,lag_return_10d,rolling_mean_5d,rolling_mean_10d,rolling_mean_20d,rolling_vol_5d,rolling_vol_10d,rolling_vol_20d,momentum_5d,momentum_10d,momentum_20d,ma_ratio_5_20,target_next_day_return
0,2015-02-03,AAPL,0.000169,0.012547,0.031133,-0.035013,0.025757,0.010113,0.011620,0.004358,0.036233,0.025471,0.023813,0.048895,0.119256,0.085063,0.042495,0.007670
1,2015-02-04,AAPL,0.007670,0.000169,-0.014634,0.056533,0.007635,0.017150,0.009061,0.005775,0.027687,0.025176,0.022587,0.087136,0.091336,0.116706,0.053734,0.007137
2,2015-02-05,AAPL,0.007137,0.007670,0.012547,0.031133,0.026015,0.007377,0.009064,0.006153,0.016790,0.025176,0.022551,0.036857,0.091374,0.125165,0.055062,-0.008421
3,2015-02-06,AAPL,-0.008421,0.007137,0.000169,-0.014634,0.005160,0.002578,0.007176,0.005809,0.010585,0.024462,0.022477,0.012728,0.071293,0.117525,0.051829,0.006642
4,2015-02-09,AAPL,0.006642,-0.008421,0.007670,0.012547,0.001062,0.003820,0.005818,0.003467,0.008142,0.024958,0.021310,0.019114,0.056819,0.067114,0.052296,0.019212


In [9]:
features_df.to_csv("data/features_large.csv", index=False)

print("Feature dataset saved as data/features_large.csv")
print("Final shape:", features_df.shape)

Feature dataset saved as data/features_large.csv
Final shape: (231849, 18)


In [10]:
features_df.columns

Index(['date', 'ticker', 'return_1d', 'lag_return_1d', 'lag_return_3d',
       'lag_return_5d', 'lag_return_10d', 'rolling_mean_5d',
       'rolling_mean_10d', 'rolling_mean_20d', 'rolling_vol_5d',
       'rolling_vol_10d', 'rolling_vol_20d', 'momentum_5d', 'momentum_10d',
       'momentum_20d', 'ma_ratio_5_20', 'target_next_day_return'],
      dtype='object')